In [2]:
!pip install sentence-transformers chromadb groq langchain langchain-community langchain-groq pandas gradio --q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\aadhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from getpass import getpass
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
if not GROQ_API_KEY:
    GROQ_API_KEY = getpass('Enter your GROQ API key:')
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY

llm=ChatGroq(model="llama-3.3-70b-versatile")

prompt=PromptTemplate(
    input_variables=[
        "Platform",
        "Content_Type",
        "Topic",
        "Context",
        "Target_Audience",
        "Tone",
        "Length",
        "Emoji_Usage",
        "Hashtag_Usage",
        "CTA",
        "Writing_Style"],
    template="""
You are an expert Social Media Content Writer and Personal Brand Strategist with experience creating high-performing content for LinkedIn, Instagram, X (Twitter), Reddit, Facebook, Threads, Medium, and other social platforms.
Your task is to create platform-native content that matches the user's tone, audience, goals, and writing style.
---
User Inputs
Platform - {Platform}

Examples:
* LinkedIn
* Instagram
* X (Twitter)
* Reddit
* Facebook
---
Content Type - {Content_Type}

Examples:
* Achievement Post
* Project Showcase
* Product Launch
* Tutorial
* Storytelling
* Personal Experience
* Career Update
* Technical Explanation
* Event Recap
* Startup Journey
* Build In Public
* Educational Post
* Opinion Post
* Community Update
---
Topic - {Topic}

---
Context / Details - {Context}

Provide all relevant details including:
* What was built
* What was achieved
* Challenges faced
* Technologies used
* Results obtained
* Lessons learned
* Key takeaways
---
Target Audience - {Target_Audience}

Examples:
* Recruiters
* Software Developers
* AI Engineers
* Startup Founders
* Students
* Data Scientists
* General Audience

---
Tone - {Tone}

Examples:

* Professional
* Casual
* Friendly
* Inspirational
* Technical
* Educational
* Storytelling
* Humorous
* Confident
* Bold

---
Length - {Length}

Options:
* Short
* Medium
* Long
* Detailed
---
Emoji Usage - {Emoji_Usage}

Options:
* Required
* Not Required
* Minimal
* Heavy
---
Hashtag Usage - {Hashtag_Usage}

Options:
* Required
* Not Required

If Required:
Generate highly relevant hashtags optimized for the selected platform.
---
Call To Action - {CTA}
Examples:

* Ask for feedback
* Invite discussion
* Encourage sharing
* Request collaboration
* No CTA

Writing Style - {Writing_Style}

Examples:
* Founder Style
* Corporate Style
* Creator Style
* Student Style
* Developer Style
* Researcher Style
* Influencer Style
---
Instructions
Generate content that:

1. Follows platform-specific best practices.
2. Matches the selected tone and style.
3. Sounds natural and human-written.
4. Avoids generic AI phrases.
5. Uses storytelling when appropriate.
6. Creates a strong hook within the first 1–3 lines.
7. Maintains readability with proper spacing.
8. Uses platform-native formatting.
9. Highlights achievements without sounding arrogant.
10. Optimizes engagement for the selected platform.
11. Includes relevant hashtags only if requested.
12. Includes emojis only if requested.
13. Includes a CTA only if requested.
14. Ensures authenticity and credibility.
15. Makes readers want to engage, comment, or share.
---
Platform-Specific Rules
LinkedIn

* Strong professional hook.
* Use short paragraphs.
* Focus on journey, lessons, impact, and results.
* Professional storytelling.
* End with a thoughtful question or CTA if requested.

Instagram

* Attention-grabbing first line.
* Conversational and engaging.
* Emotional or relatable storytelling.
* Suitable for captions.
* Include hashtag section if requested.

X (Twitter)

* Concise and impactful.
* Hook immediately.
* Optimize for reposts and engagement.
* Use thread format when content is long.

Reddit

* Authentic and community-focused.
* Avoid promotional language.
* Prioritize value and discussion.
* Match subreddit culture.
* Sound like a real community member.

Facebook

* Conversational and community-oriented.
* Focus on storytelling and engagement.
---

Output Format

Content

[Generated Content]

Suggested Hook Variations

1. ...
2. ...
3. ...

 Hashtags

[Only if requested]

 Engagement Score Strategy

* Why this content works
* Expected audience reaction
* Engagement optimization techniques used
"""
)
parser=StrOutputParser()
chain=prompt |llm|parser

# response = chain.invoke({
#     "Platform": "LinkedIn",
#     "Content_Type": "Project Showcase",
#     "Topic": "Calculator App using Streamlit",
#     "Context": "Built a calculator that accepts multiple inputs and performs arithmetic operations using Streamlit.",
#     "Target_Audience": "Software Developers",
#     "Tone": "Professional",
#     "Length": "Long",
#     "Emoji_Usage": "Minimal",
#     "Hashtag_Usage": "Required",
#     "CTA": "Ask for feedback",
#     "Writing_Style": "Developer Style"
# })

# print(response)

import gradio as gr


def generate_content(
    Platform,
    Content_Type,
    Topic,
    Context,
    Target_Audience,
    Tone,
    Length,
    Emoji_Usage,
    Hashtag_Usage,
    CTA,
    Writing_Style
):

    response = chain.invoke({
        "Platform": Platform,
        "Content_Type": Content_Type,
        "Topic": Topic,
        "Context": Context,
        "Target_Audience": Target_Audience,
        "Tone": Tone,
        "Length": Length,
        "Emoji_Usage": Emoji_Usage,
        "Hashtag_Usage": Hashtag_Usage,
        "CTA": CTA,
        "Writing_Style": Writing_Style
    })

    return response


custom_css = """
.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
}

footer {
    display: none !important;
}
"""


with gr.Blocks(
    title="Social Media Content Generator",
    theme=gr.themes.Monochrome(),
    css=custom_css
) as demo:

    gr.Markdown("# Social Media Content Generator")
    gr.Markdown("Generate platform-specific social media content.")

    with gr.Row():
        Platform = gr.Dropdown(
            ["LinkedIn", "Instagram", "X", "Reddit", "Facebook"],
            label="Platform"
        )

        Content_Type = gr.Dropdown(
            [
                "Achievement Post",
                "Project Showcase",
                "Product Launch",
                "Tutorial",
                "Storytelling",
                "Personal Experience",
                "Career Update",
                "Technical Explanation",
                "Event Recap",
                "Startup Journey",
                "Build In Public",
                "Educational Post",
                "Opinion Post",
                "Community Update"
            ],
            label="Content Type"
        )

    Topic = gr.Textbox(
        label="Topic",
        placeholder="Enter your topic"
    )

    Context = gr.Textbox(
        label="Context / Details",
        lines=8,
        placeholder="Describe your project, achievement, experience, etc."
    )

    with gr.Row():
        Target_Audience = gr.Textbox(
            label="Target Audience"
        )

        Tone = gr.Dropdown(
            [
                "Professional",
                "Casual",
                "Friendly",
                "Inspirational",
                "Technical",
                "Educational",
                "Storytelling",
                "Humorous",
                "Confident",
                "Bold"
            ],
            label="Tone"
        )

    with gr.Row():

        Length = gr.Dropdown(
            ["Short", "Medium", "Long", "Detailed"],
            label="Length"
        )

        Emoji_Usage = gr.Dropdown(
            ["Required", "Minimal", "Heavy", "Not Required"],
            label="Emoji Usage"
        )

        Hashtag_Usage = gr.Dropdown(
            ["Required", "Not Required"],
            label="Hashtag Usage"
        )

    CTA = gr.Textbox(
        label="Call To Action"
    )

    Writing_Style = gr.Dropdown(
        [
            "Founder Style",
            "Corporate Style",
            "Creator Style",
            "Student Style",
            "Developer Style",
            "Researcher Style",
            "Influencer Style"
        ],
        label="Writing Style"
    )

    generate_btn = gr.Button(
        "Generate Content",
        variant="primary"
    )

    output = gr.Markdown(
        label="Generated Content"
    )

    generate_btn.click(
        fn=generate_content,
        inputs=[
            Platform,
            Content_Type,
            Topic,
            Context,
            Target_Audience,
            Tone,
            Length,
            Emoji_Usage,
            Hashtag_Usage,
            CTA,
            Writing_Style
        ],
        outputs=output
    )


demo.launch(share=True)

C:\Users\aadhi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\aadhi\AppData\Local\Temp\ipykernel_16280\508416275.py:296: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://6080ea220fa6e558df.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
